In [0]:
### Step 1: Initializing the spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
spark= SparkSession.builder.appName("green_grid_case_study").getOrCreate()

In [0]:
### Step 2: Reading the data residing in the volume under the bronze_layer schema. This serves as our staging layer where the raw data lie sin its original form
subscribers_data= spark.read.format("csv").option("header","true").option("inferSchema","true").option("mode","PERMISSIVE").load("/Volumes/santeflux_case_study_3/bronze_layer/raw_data_volume/Dataset_subs.csv")
subscribers_vitals_data= spark.read.format("csv").option("header","true").option("inferSchema","true").option("mode","PERMISSIVE").load("/Volumes/santeflux_case_study_3/bronze_layer/raw_data_volume/Dataset_vitals.csv")

In [0]:
### Step 3: We first handedly have to implement the MASKING of the cruical perosnal information as mentioned in the business requirements.
### The masking output should have the following format "Maris W****". But how we will do this? 
#### We will use SPLIT+REPEAT+CONCAT funtions in combination to mask PII (names) directly at the source, preventing sensitive data from ever reaching the Silver or Gold storage layers.

subscribers_vitals_data= subscribers_vitals_data.withColumn("Full_Name_New",f.concat(f.split(f.col("Full_Name")," ")[0],f.lit(" "),f.substring(f.split(f.col("Full_Name")," ")[1],1,1),f.repeat(f.lit("*"),f.length(f.split(f.col("Full_Name")," ")[1])-1)))


In [0]:
subscribers_vitals_data.show(3)

In [0]:
#### Step 4: After masking our relevant data, we have to push this data back into our bronze_layer as a table so that we can proceed further with the ETl steps.

subscribers_vitals_data.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("santeflux_case_study_3.bronze_layer.raw_subscribers_vitals_data") 

In [0]:
#### Step 5: The next task is to filter out the GHOST USERS so that we only have the relevant data later to perform a join
#### For this process we have to use WINDOW FUNCTION 

### Reading our bronze table for our silver layer and ead only the relvant columns 
subscribers_vitals_data_silver =spark.read.table("santeflux_case_study_3.bronze_layer.raw_subscribers_vitals_data").select(f.col("User_ID").alias("user_id"),f.col("Full_Name_New").alias("full_name"), f.col("Timestamp").alias("timestamp"),f.col("ville"),f.col("Heart_Rate").alias("heart_rate"),f.col("Status").alias("status"))

### Next we have to ensure that our datatypes are in place
subscribers_vitals_data_silver=subscribers_vitals_data_silver.withColumn("timestamp",f.to_timestamp(f.col("Timestamp")))
subscribers_vitals_data_silver=subscribers_vitals_data_silver.withColumn("ville",f.initcap(f.col("ville")))

### Performing the frozen record detection strategy. For this we will create a 10 minute window and monitro the max and min heart rate over this 10 minute window created
from pyspark.sql.window import Window
subscribers_vitals_data_silver=subscribers_vitals_data_silver.withColumn("frozen_rn", f.window(f.col("timestamp"),"10 minutes"))
subscribers_vitals_data_silver.filter(f.col("user_id")==2045).show(20)

### From here we can see that actually those records that were frozen had a continuous running window for straight 10 minutes

In [0]:
#### Step 6: Now we have to separate the frozen records from the table and filter the records we want for the further processing. Now that we have filtered our frozen records we have to store them separately in our silver layer as a table in the catalog.

frozen_records= subscribers_vitals_data_silver.filter((f.col("status").like("%FROZEN%"))).drop(f.col("frozen_rn"))

frozen_records.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("santeflux_case_study_3.silver_layer.fact_frozen_records")

### In the same step, we have to filter out these records form our orginal dataframe
subscribers_vitals_data_silver_final = subscribers_vitals_data_silver.filter(~f.col("status").like("%FROZEN%")).drop("frozen_rn")

In [0]:
### Step 7: After we have our valid data we now have to gind GHOST USRS. Those who are present in the subscribers_vital data but haven't actuallly purchased any subscription. For this we have to perform the BROADCAST JOIN STRATEGY as mentioned in the business requirements.Alogn with the broadcast JOIN TECHNIQUE we have to use LEFT_aNTI join to filter our ghost users

joined_df= subscribers_vitals_data_silver_final.alias("a").join(f.broadcast(subscribers_data).alias("b"), on=f.col("a.user_id")==f.col("b.User_ID"), how="left_anti")

ghost_users=joined_df.select(f.col("user_id"),f.col("full_name")).dropDuplicates()
ghost_users.show(5)

### These ghost_users_data now needs to be written back in our catalog
ghost_users.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("santeflux_case_study_3.silver_layer.dim_ghost_users") 

In [0]:
#### Step 8: Now that we have seperated our ghost users we can now have our final data with us. To filter the final data we have to use another LEFT ANTI join this time with the ghost_users and the subscribers_vitals_data_silver_final table

fact_subscribers_vitals_data= subscribers_vitals_data_silver_final.alias("a").join(f.broadcast(ghost_users).alias("b"), on=f.col("a.user_id")==f.col("b.user_id"),how="left_anti")

fact_subscribers_vitals_data= fact_subscribers_vitals_data.alias("a").join(subscribers_data.alias("b"),on=f.col("a.user_id")==f.col("b.User_ID"),how="left").drop(f.col("b.User_ID"))

fact_subscribers_vitals_data.show(5)


In [0]:
### After the same, we have to push our final filtered data back into the catalog as FACT_SUBSCRIBERS_VITALS_DATA
fact_subscribers_vitals_data.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("santeflux_case_study_3.silver_layer.fact_subscribers_vitals_data")

In [0]:
### Step 9: The Gold Layer has to be built that contains the final data that is required for the analysis. The aggregated table should showcase aggreagtes of heart rate trends by location and subcription tier

agg_dataframe= spark.read.table("santeflux_case_study_3.silver_layer.fact_subscribers_vitals_data").select(f.col("user_id"),f.col("full_name"),f.col("timestamp"),f.col("ville"),f.col("heart_rate"),f.col("status"),f.col("Subscription_Type").alias("subscription_type"))

final_agg_data= agg_dataframe.groupBy(f.col("ville"),f.col("subscription_type")).agg(f.round(f.avg(f.col("heart_rate")),2).alias("average_heart_rate"),f.max(f.col("heart_rate")).alias("max_heart_rate"),f.min(f.col("heart_rate")).alias("min_heart_rate")).orderBy(f.col("ville"))

final_agg_data.show()

In [0]:
### Step 10: Lastly we will be savign this table in our gold layer for dashboarding
final_agg_data.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("santeflux_case_study_3.gold_layer.fact_aggregated_data")